# Unknown test set, step 1 — build per-sample folders

Reorganizes the independent unknown-concentration test spectra into one folder
per sample, where a sample is one variant–concentration pair, and writes each as
an ML-format CSV interpolated onto the common wavenumber grid.

**Input** — the raw unknown-test CSVs.

**Output** — one subfolder per sample, each holding one ML-format CSV.

**Next** — `02_recover_unknown_test_samples.ipynb`.


In [ ]:
# ============================================================
# STEP 0: Group CSVs into sample folders
# ============================================================
import os
import re
import shutil

root = r"/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/Unknown_test_with_sequential_dilution_01scale_mean"

# --- Parse all CSVs in root folder ---
# Expected patterns:
#   Base:   {virus}-{conc1}.csv
#   Dil 1:  {virus}-{conc1}-D1-{conc2}.csv
#   Dil 2:  {virus}-{conc1}-D2-{conc3}.csv

pattern_base = re.compile(r'^(.+?)-(\d+(?:\.\d+)?)\.csv$')
pattern_d1   = re.compile(r'^(.+?)-(\d+(?:\.\d+)?)-D1-(\d+(?:\.\d+)?)\.csv$')
pattern_d2   = re.compile(r'^(.+?)-(\d+(?:\.\d+)?)-D2-(\d+(?:\.\d+)?)\.csv$')

files = [f for f in os.listdir(root) if f.endswith('.csv')]

# Build lookup dictionaries keyed by (virus, conc_str)
base_files = {}   # (virus, conc1) -> filename
d1_files   = {}   # (virus, conc1) -> (filename, conc2)
d2_files   = {}   # (virus, conc1) -> (filename, conc3)

for f in files:
    m = pattern_d2.match(f)
    if m:
        virus, conc1, conc3 = m.group(1), m.group(2), m.group(3)
        d2_files[(virus, conc1)] = (f, conc3)
        continue
    m = pattern_d1.match(f)
    if m:
        virus, conc1, conc2 = m.group(1), m.group(2), m.group(3)
        d1_files[(virus, conc1)] = (f, conc2)
        continue
    m = pattern_base.match(f)
    if m:
        virus, conc1 = m.group(1), m.group(2)
        base_files[(virus, conc1)] = f

# --- Build triplets and create sample folders ---
sample_counter = 1
sample_registry = []  # will be used in Step 2

for (virus, conc1), base_fname in sorted(base_files.items()):
    if (virus, conc1) not in d1_files:
        print(f"[WARN] No D1 file for {virus}-{conc1}, skipping.")
        continue
    d1_fname, conc2 = d1_files[(virus, conc1)]

    if (virus, conc1) not in d2_files:
        print(f"[WARN] No D2 file for {virus}-{conc1}, skipping.")
        continue
    d2_fname, conc3 = d2_files[(virus, conc1)]

    # Helper: format concentration label (drop trailing .0 if integer-valued)
    def fmt_conc(c):
        f = float(c)
        return str(int(f)) if f == int(f) else str(f)

    # --- Sample A: base + D1 ---
    folder_a = f"sample{sample_counter}_{virus}-{fmt_conc(conc1)}-{fmt_conc(conc2)}"
    path_a = os.path.join(root, folder_a)
    os.makedirs(path_a, exist_ok=True)
    shutil.copy(os.path.join(root, base_fname), path_a)
    shutil.copy(os.path.join(root, d1_fname),   path_a)
    sample_registry.append({
        'folder': folder_a,
        'path': path_a,
        'virus': virus,
        'files': [
            {'fname': base_fname, 'conc': float(conc1)},
            {'fname': d1_fname,   'conc': float(conc2)},
        ]
    })
    print(f"Created: {folder_a}")
    sample_counter += 1

    # --- Sample B: D1 + D2 ---
    folder_b = f"sample{sample_counter}_{virus}-{fmt_conc(conc2)}-{fmt_conc(conc3)}"
    path_b = os.path.join(root, folder_b)
    os.makedirs(path_b, exist_ok=True)
    shutil.copy(os.path.join(root, d1_fname), path_b)
    shutil.copy(os.path.join(root, d2_fname), path_b)
    sample_registry.append({
        'folder': folder_b,
        'path': path_b,
        'virus': virus,
        'files': [
            {'fname': d1_fname, 'conc': float(conc2)},
            {'fname': d2_fname, 'conc': float(conc3)},
        ]
    })
    print(f"Created: {folder_b}")
    sample_counter += 1

print(f"\nTotal samples created: {len(sample_registry)}")

In [ ]:
# ============================================================
# STEP 1 & 2: Combine CSVs into ML-format CSV per sample
# ============================================================
import pandas as pd
import numpy as np

# === USER CONFIG ===
DATE_STR = "03062026"        # <-- Set your date in MMDDYYYY format
WAVENUMBER_START = 400
WAVENUMBER_END   = 1800
# ===================

def load_spectra_csv(filepath):
    """
    Load a tab-separated SERS CSV.
    Returns a DataFrame with rows = spectra, columns = wavenumbers (numeric).
    Automatically detects the wavenumber column by name (case-insensitive).
    """
    df = pd.read_csv(filepath, sep='\t')

    # Identify wavenumber column (flexible spelling)
    wn_col = None
    for col in df.columns:
        if 'wave' in col.lower() or 'wn' in col.lower() or 'raman' in col.lower():
            wn_col = col
            break
    if wn_col is None:
        raise ValueError(f"Cannot identify wavenumber column in {filepath}. Columns: {list(df.columns)}")

    wavenumbers = df[wn_col].values
    spectra_cols = [c for c in df.columns if c != wn_col]
    spectra_matrix = df[spectra_cols].values  # shape: (n_wavenumbers, n_spectra)

    return wavenumbers, spectra_matrix  # spectra_matrix.T gives (n_spectra, n_wavenumbers)


def interpolate_to_grid(wavenumbers, spectra_matrix, wn_start, wn_end):
    """
    Interpolate each spectrum onto an integer wavenumber grid [wn_start, wn_end].
    spectra_matrix shape: (n_wavenumbers, n_spectra)
    Returns: (grid, interpolated_matrix) where interpolated_matrix is (n_spectra, n_grid)
    """
    grid = np.arange(wn_start, wn_end + 1, dtype=float)
    n_spectra = spectra_matrix.shape[1]
    result = np.zeros((n_spectra, len(grid)))
    for i in range(n_spectra):
        result[i, :] = np.interp(grid, wavenumbers, spectra_matrix[:, i])
    return grid, result


def build_ml_row(intensity_row, grid, virus_name, conc):
    """
    Build a single ML-format row dict.
    Wavenumber columns: intensity rounded to 3 decimal places.
    Label: "['virus_name']"  — note: string representation of a Python list
    Conc:  "[conc_float]"
    """
    row = {str(int(wn)): round(val, 3) for wn, val in zip(grid, intensity_row)}
    row['Label'] = f"['{virus_name}']"
    row['Conc']  = f"[{float(conc):.1f}]"
    return row


# --- Process each sample ---
for sample in sample_registry:
    rows = []
    grid = None

    for file_info in sample['files']:
        fpath = os.path.join(sample['path'], file_info['fname'])
        conc  = file_info['conc']
        virus = sample['virus']

        wavenumbers, spectra_matrix = load_spectra_csv(fpath)
        g, interp = interpolate_to_grid(wavenumbers, spectra_matrix,
                                        WAVENUMBER_START, WAVENUMBER_END)
        if grid is None:
            grid = g

        for i in range(interp.shape[0]):
            rows.append(build_ml_row(interp[i], grid, virus, conc))

    # Assemble DataFrame — column order: wavenumbers, Label, Conc
    wn_cols = [str(int(w)) for w in grid]
    ml_df = pd.DataFrame(rows, columns=wn_cols + ['Label', 'Conc'])

    out_fname = f"{DATE_STR}-{sample['folder']}-ML_format.csv"
    out_path  = os.path.join(sample['path'], out_fname)
    ml_df.to_csv(out_path, index=False)
    print(f"Saved: {out_fname}  ({len(ml_df)} spectra)")

print("\nAll ML-format CSVs generated.")